# [Super AI Engineer Season 6] Mini Hackathon 2 Level 2
## Parasite Eggs

**Super AI Engineer Season 6 - Level 2 Hackathon**  
- Dataset: Chula-ParasiteEgg-11 vs SPAI Test Set
- จัดทำโดย: 600425-วิศิษฐ์

---
### Notebook Outline
1. Setup & Imports  
2. Data Loading & Robust Indexing  
3. Extreme Synthetic Copy-Paste & Target Domain Negative Sampling
4. YOLOv8 Dataset Configuration
5. Model Training (YOLOv8s)
6. Inference with Test-Time Augmentation (TTA)
7. Prediction & Submission Generation

---
### Domain Shift Handling Techniques (ICIP 2022 Challenge Framework)
- **Extreme Synthetic Copy-Paste**: สร้างพื้นหลังจำลองที่คล้ายคลึงกับ Test Set มากที่สุด (เพิ่มขอบดำจากเลนส์กล้องจุลทรรศน์, เส้นสเกลไม้บรรทัด และปรับย้อมสีภาพ)
- **Target Domain Negative Sampling**: ใช้ภาพจาก Test Set นำมาผ่านกระบวนการ Heavy Blur เพื่อลบโครงสร้างไข่พยาธิ และใช้เป็น Negative Sample ให้โมเดลเรียนรู้สัญญาณรบกวนในโดเมนเป้าหมาย
- **Object Detection (YOLOv8s)**: ใช้โมเดลการหาตำแหน่งวัตถุ เพื่อป้องกันปัญหา False Positive จากสัญญาณรบกวนรอบนอก
- **Test-Time Augmentation (TTA)**: เปิดใช้งานระหว่างการ Inference เพื่อปรับปรุงประสิทธิภาพการตรวจจับวัตถุขนาดเล็ก

# 1. Setup & Imports
### 1.1 นำเข้าไลบรารีที่จำเป็น

In [1]:
!pip install -q ultralytics

import os, random, gc, shutil, warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import cv2
from pathlib import Path
from PIL import Image, ImageEnhance, ImageFilter
import matplotlib.pyplot as plt
from tqdm.notebook import tqdm
from ultralytics import YOLO

# ตั้งค่า Seed เพื่อให้ผลลัพธ์ Reproducible
SEED = 42
random.seed(SEED); np.random.seed(SEED)

CLASSES = [
    "Ascaris lumbricoides",       # 0
    "Capillaria philippinensis",  # 1
    "Enterobius vermicularis",    # 2
    "Fasciolopsis buski",         # 3
    "Hookworm egg",               # 4
    "Hymenolepis diminuta",       # 5
    "Hymenolepis nana",           # 6
    "Opisthorchis viverrine",     # 7
    "Paragonimus spp",            # 8
    "Taenia spp. egg",            # 9
    "Trichuris trichiura",        # 10
]

# Mapping คีย์เวิร์ดสำหรับค้นหาไฟล์ ป้องกันปัญหาโฟลเดอร์ไม่ตรงตามรูปแบบ
CLASS_KEYWORDS = {
    "ascaris": 0,
    "capillaria": 1,
    "enterobius": 2,
    "fasciolopsis": 3,
    "hookworm": 4,
    "diminuta": 5,
    "nana": 6,
    "opisthorchis": 7,
    "paragonimus": 8,
    "taenia": 9,
    "trichuris": 10
}

WORK_DIR = Path('/kaggle/working')
SAMPLE_SUB = None

# ค้นหาไฟล์ Sample Submission อย่างไดนามิก
for p in Path('/kaggle/input').rglob('sample_submission.csv'):
    SAMPLE_SUB = p
    break

# ค้นหาไฟล์ Test Set โดยอ้างอิงจากตำแหน่งของ Sample Submission
test_files_for_bg = []
if SAMPLE_SUB is not None:
    comp_root = SAMPLE_SUB.parent
    all_comp_images = sorted(list(comp_root.rglob('*.png')) + list(comp_root.rglob('*.jpg')))
    test_files_for_bg = [f for f in all_comp_images if 'test' in str(f).lower()]

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 19.7 MB/s eta 0:00:00a 0:00:01
Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.


# 2. Data Loading & Robust Indexing
### 2.1 ค้นหาไฟล์ภาพไข่พยาธิทั้งหมดในระบบ Kaggle อัตโนมัติ

In [2]:
print('[Phase 1/5] Indexing Dataset Images...')
train_image_paths = []
train_labels = []

search_root = Path('/kaggle/input')
if search_root.exists():
    for p in search_root.rglob('*.*'):
        if p.suffix.lower() in ['.jpg', '.png', '.jpeg']:
            # ข้ามโฟลเดอร์ Test Set เพื่อไม่ให้ข้อมูลปะปนกัน
            if 'test_set' in p.parts or 'super-ai-engineer' in p.parts:
                continue
            
            full_str = str(p).lower().replace('_', ' ')
            for kw, idx in CLASS_KEYWORDS.items():
                if kw in full_str:
                    train_image_paths.append(p)
                    train_labels.append(idx)
                    break

if len(train_image_paths) == 0:
    print('CRITICAL ERROR: Dataset not found!')
    print('Please add the "macharning/chula-parasite-dataset" to the notebook environment.')
else:
    print(f'Successfully indexed {len(train_image_paths)} parasite egg images.')

# สุ่มลำดับข้อมูล
combined = list(zip(train_image_paths, train_labels))
random.shuffle(combined)
train_image_paths, train_labels = zip(*combined)
train_image_paths = list(train_image_paths)
train_labels = list(train_labels)

[Phase 1/5] Indexing Dataset Images...
Successfully indexed 11000 parasite egg images.


# 3. Extreme Synthetic Copy-Paste & Negative Sampling
### 3.1 สร้างโครงสร้างข้อมูลสำหรับ YOLOv8

In [3]:
yolo_dir = WORK_DIR / 'yolo_dataset'
for split in ['train', 'val']:
    os.makedirs(yolo_dir / f'images/{split}', exist_ok=True)
    os.makedirs(yolo_dir / f'labels/{split}', exist_ok=True)
    
yaml_content = f"""path: {yolo_dir.absolute()}
train: images/train
val: images/val

names:
"""
for i, c in enumerate(CLASSES):
    yaml_content += f"  {i}: '{c}'\n"
    
with open(WORK_DIR / 'parasite.yaml', 'w') as f:
    f.write(yaml_content)

### 3.2 สร้างภาพจำลองระดับสูง (Advanced Domain Matching & Augmentation)

In [4]:
def create_extreme_synthetic_bg():
    """สร้างภาพพื้นหลังที่จำลองลักษณะเฉพาะของ Test Domain"""
    bg_type = random.choice(['microscope', 'pink', 'yellow', 'brown'])
    
    if bg_type == 'microscope':
        c = (int(np.random.normal(233, 10)), int(np.random.normal(226, 10)), int(np.random.normal(202, 10)))
    elif bg_type == 'pink':
        c = (255, random.randint(160, 210), random.randint(160, 210))
    elif bg_type == 'yellow':
        c = (255, 255, random.randint(100, 180))
    else:
        c = (random.randint(180, 220), random.randint(150, 190), random.randint(100, 150))
        
    bg = np.full((500, 500, 3), c, dtype=np.uint8)
    
    noise = np.random.randint(0, 30, (500, 500, 3), dtype=np.uint8)
    bg = np.clip(bg - noise, 0, 255).astype(np.uint8)
    
    # จำลองเส้นสเกลไม้บรรทัด (Graticule) ที่พบใน Test Set
    if random.random() < 0.25:
        y_pos = random.randint(50, 450)
        cv2.line(bg, (0, y_pos), (500, y_pos), (30, 30, 30), 1)
        for x in range(0, 500, 10):
            h = 15 if x % 50 == 0 else 5
            cv2.line(bg, (x, y_pos), (x, y_pos - h), (30, 30, 30), 1)
            
    # จำลองขอบสีดำของเลนส์กล้องจุลทรรศน์ (Eyepiece Border)
    if random.random() < 0.35:
        center_x = random.randint(50, 450)
        center_y = random.randint(50, 450)
        radius = random.randint(250, 400)
        mask = np.zeros((500, 500), dtype=np.uint8)
        cv2.circle(mask, (center_x, center_y), radius, 255, -1)
        mask = cv2.GaussianBlur(mask, (51, 51), 0)
        mask_norm = mask / 255.0
        bg = (bg * mask_norm[..., np.newaxis]).astype(np.uint8)
        
    return Image.fromarray(bg)

print('[Phase 2/5] Generating Domain-Matched Synthetic Images & Target Negative Samples...')

split_idx = int(len(train_image_paths) * 0.85)

for idx_loop, (p, cls_idx) in enumerate(tqdm(zip(train_image_paths, train_labels), total=len(train_image_paths))):
    split = 'train' if idx_loop < split_idx else 'val'
    
    bg = create_extreme_synthetic_bg()
    
    try:
        egg = Image.open(p).convert('RGBA')
    except:
        continue
        
    egg = egg.rotate(random.randint(0, 360), expand=True, fillcolor=(255,255,255,0))
    scale = random.uniform(0.7, 1.4)
    new_w, new_h = int(egg.width * scale), int(egg.height * scale)
    egg = egg.resize((new_w, new_h))
    
    enhancer = ImageEnhance.Brightness(egg)
    egg = enhancer.enhance(random.uniform(0.7, 1.3))
    
    max_x = 500 - new_w
    max_y = 500 - new_h
    
    if max_x <= 0 or max_y <= 0:
        egg.thumbnail((400, 400), Image.Resampling.LANCZOS)
        new_w, new_h = egg.size
        max_x = 500 - new_w
        max_y = 500 - new_h
        
    paste_x = random.randint(0, max_x)
    paste_y = random.randint(0, max_y)
    
    bg.paste(egg, (paste_x, paste_y), mask=egg)
    
    x_center = (paste_x + new_w / 2) / 500.0
    y_center = (paste_y + new_h / 2) / 500.0
    w_norm = new_w / 500.0
    h_norm = new_h / 500.0
    
    img_name = f"syn_{split}_{idx_loop}.jpg"
    bg.save(yolo_dir / f'images/{split}' / img_name, quality=90)
    
    with open(yolo_dir / f'labels/{split}/syn_{split}_{idx_loop}.txt', 'w') as f:
        f.write(f"{cls_idx} {x_center:.6f} {y_center:.6f} {w_norm:.6f} {h_norm:.6f}\n")

# ─── Target Domain Negative Sampling ───
print('Generating Target Domain Negative Samples...')
num_negatives = min(1000, len(test_files_for_bg))
sampled_test_files = random.sample(test_files_for_bg, num_negatives) if num_negatives > 0 else []

for i, p in enumerate(sampled_test_files):
    split = 'train' if i < int(num_negatives * 0.85) else 'val'
    try:
        # ดึงภาพจาก Test Set ทำ Heavy Blur เพื่อใช้เป็นภาพพื้นหลังจำลอง (Negative Data)
        bg = Image.open(p).convert('RGB').resize((500, 500))
        bg = bg.filter(ImageFilter.GaussianBlur(radius=8))
    except:
        continue
        
    img_name = f"neg_target_{split}_{i}.jpg"
    bg.save(yolo_dir / f'images/{split}' / img_name, quality=90)
    # สร้างไฟล์ฉลากเปล่า เพื่อระบุว่าไม่มีวัตถุที่ต้องการตรวจจับ
    open(yolo_dir / f'labels/{split}/neg_target_{split}_{i}.txt', 'w').close()
    
print('Domain-Matched Synthetic Dataset Ready.')

[Phase 2/5] Generating Domain-Matched Synthetic Images & Target Negative Samples...


  0%|          | 0/11000 [00:00<?, ?it/s]

Generating Target Domain Negative Samples...
Domain-Matched Synthetic Dataset Ready.


# 4. Model Training (YOLOv8s)
### 4.1 เทรนโมเดลสำหรับการตรวจจับวัตถุ

In [5]:
print('[Phase 3/5] Training YOLOv8s Model...')
model = YOLO('yolov8s.pt')  

# ปรับ Batch Size เป็น 16 เพื่อป้องกันปัญหา CUDA Out of Memory อย่างเด็ดขาด
results = model.train(
    data=str(WORK_DIR / 'parasite.yaml'),
    epochs=25,
    imgsz=500,
    batch=16,
    project='parasite_yolo',
    name='run_sota',
    optimizer='AdamW',
    lr0=0.001,
    patience=10,
    augment=True,
    mosaic=1.0,
    hsv_h=0.15, hsv_s=0.6, hsv_v=0.4,
    degrees=90.0,
    flipud=0.5, fliplr=0.5
)

[Phase 3/5] Training YOLOv8s Model...
Ultralytics 8.4.41 🚀 Python-3.12.12 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=True, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/kaggle/working/parasite.yaml, degrees=90.0, deterministic=True, device=None, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=25, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.5, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.15, hsv_s=0.6, hsv_v=0.4, imgsz=500, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.001, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8s.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=run_sota, nbs=64, nms=False, opset=None, optimize=False, optimiz

# 5. Inference with Test-Time Augmentation (TTA)
### 5.1 ดำเนินการทดสอบบน Test Set พร้อมเปิดใช้งาน TTA

In [7]:
print('[Phase 4/5] Running Inference with TTA...')

# แก้ไขด่วน: กวาดหา best.pt ทั่วระบบ Kaggle ไม่ให้หลุดรอด
best_models = list(Path('/kaggle/working').rglob('best.pt'))
if len(best_models) > 0:
    # ดึงไฟล์ที่เพิ่งสร้างล่าสุดมาใช้
    best_model_path = sorted(best_models, key=os.path.getmtime)[-1]
    trained_model = YOLO(best_model_path)
    print(f"Successfully loaded best model from: {best_model_path}")
else:
    raise FileNotFoundError("CRITICAL ERROR: YOLO training failed (best.pt not found).")

final_predictions = {}
CONF_THRESHOLD = 0.15

if len(test_files_for_bg) > 0:
    for path in tqdm(test_files_for_bg):
        res = trained_model.predict(path, imgsz=500, conf=CONF_THRESHOLD, augment=True, verbose=False)[0]
        
        if len(res.boxes) > 0:
            best_box_idx = np.argmax(res.boxes.conf.cpu().numpy())
            best_box = res.boxes[best_box_idx]
            pred_class = int(best_box.cls.item())
            
            final_predictions[path.name] = pred_class
        else:
            final_predictions[path.name] = -1
else:
    print("Error: No test images found for inference.")

[Phase 4/5] Running Inference with TTA...
Successfully loaded best model from: /kaggle/working/runs/detect/parasite_yolo/run_sota/weights/best.pt


  0%|          | 0/2002 [00:00<?, ?it/s]

WARNING ⚠️ imgsz=[500] must be multiple of max stride 32, updating to [512]
WARNING ⚠️ imgsz=[500] must be multiple of max stride 32, updating to [512]
WARNING ⚠️ imgsz=[500] must be multiple of max stride 32, updating to [512]
WARNING ⚠️ imgsz=[500] must be multiple of max stride 32, updating to [512]
WARNING ⚠️ imgsz=[500] must be multiple of max stride 32, updating to [512]
WARNING ⚠️ imgsz=[500] must be multiple of max stride 32, updating to [512]
WARNING ⚠️ imgsz=[500] must be multiple of max stride 32, updating to [512]
WARNING ⚠️ imgsz=[500] must be multiple of max stride 32, updating to [512]
WARNING ⚠️ imgsz=[500] must be multiple of max stride 32, updating to [512]
WARNING ⚠️ imgsz=[500] must be multiple of max stride 32, updating to [512]
WARNING ⚠️ imgsz=[500] must be multiple of max stride 32, updating to [512]
WARNING ⚠️ imgsz=[500] must be multiple of max stride 32, updating to [512]
WARNING ⚠️ imgsz=[500] must be multiple of max stride 32, updating to [512]
WARNING ⚠️ i

# 6. Prediction & Submission Generation
### 6.1 บันทึกไฟล์ผลลัพธ์

In [8]:
print('[Phase 5/5] Generating Submission File...')
if SAMPLE_SUB is not None and SAMPLE_SUB.exists() and len(final_predictions) > 0:
    sub = pd.read_csv(SAMPLE_SUB)
    sub['label'] = sub['filename'].map(final_predictions)
    sub['label'] = sub['label'].fillna(-1).astype(int)
    sub.to_csv('submission_yolov8s_sota_domain_shift.csv', index=False)
    
    print("Saved 'submission_yolov8s_sota_domain_shift.csv' successfully.")
    
    egg_count = (sub['label'] != -1).sum()
    bg_count = (sub['label'] == -1).sum()
    print(f"Statistics:")
    print(f"   - Detected Parasite Eggs : {egg_count} images")
    print(f"   - Rejected as Background : {bg_count} images")
    
    detected_sub = sub[sub['label'] != -1]
    if len(detected_sub) > 0:
        plt.figure(figsize=(10, 5))
        ax = sns.countplot(data=detected_sub, x='label', palette='viridis')
        ax.set_title('Predicted Class Distribution (Excluding Background -1)', fontweight='bold')
        ax.set_xticklabels([CLASSES[int(x.get_text())] for x in ax.get_xticklabels()], rotation=45, ha='right')
        plt.tight_layout()
        plt.show()
else:
    print("Error: Cannot generate submission. Sample submission file not found.")

[Phase 5/5] Generating Submission File...
Saved 'submission_yolov8s_sota_domain_shift.csv' successfully.
Statistics:
   - Detected Parasite Eggs : 1851 images
   - Rejected as Background : 149 images


NameError: name 'sns' is not defined

<Figure size 1000x500 with 0 Axes>